# SelectOmics showcase · TCGA ovarian carcinoma

What the package does, what each evaluation module reports, and how the result
compares to the obvious alternatives. Start with
`../GS-GBM/GBM_Quickstart.ipynb` if you have not run the pipeline before.

**Dataset.** 284 samples, four omic layers, 4 subtypes at a 1.23:1 imbalance.
Balanced classes are the reason this cohort is used here: method comparisons in
Sections 6 and 7 are not confounded by one arm handling imbalance better.

| Layer | Features | n/p |
|---|---|---|
| miRNA | 321 | 0.88 |
| CNV | 11,205 | 0.025 |
| Methy | 11,191 | 0.025 |
| mRNA | 11,344 | 0.025 |
| **merged** | **34,061** | **0.008** |

The merged layer at n/p = 0.008 is the regime the package is built for.

## Contents

| Section | Shows |
|---|---|
| 3 | A full run with every evaluation module enabled |
| 4 | Each evaluation module and what it reports |
| 5 | The recommendation, and when to override the last step |
| 6 | Panel size against LASSO, ElasticNet and RFECV at matched AUC |
| 7 | Step ablation: what each step contributes alone |
| 8 | Multi-omics integration: per-layer against merged |
| 9 | Reproducibility and checkpoint resume |

## Runtime

`RUN_SCALE` in Section 1 controls this.

- `"quick"` (default) uses miRNA, 321 features. **About 20 minutes** for the
  whole notebook.
- `"full"` adds the three wide layers and the merged matrix. **Several hours**;
  the merged layer alone took roughly two hours in the benchmark runs.

Every section works at either scale. The conclusions in Sections 6 to 8 were
measured at full scale and are quoted where the quick run cannot reproduce them.

## 1 · Setup

In [ ]:
import subprocess
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

_pkg_root = str(Path("../..").resolve())
if _pkg_root not in sys.path:
    sys.path.insert(0, _pkg_root)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", _pkg_root],
               capture_output=True, text=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import SelectOmics
from SelectOmics import SelectOmicsConfig, SelectOmicsPipeline

SelectOmics.enable_logging("INFO")

# "quick" -> miRNA only (321 features, ~20 min total)
# "full"  -> every layer plus the merged matrix (several hours)
RUN_SCALE = "quick"

PRIMARY_LAYER = "miRNA" if RUN_SCALE == "quick" else "mRNA"
ALL_LAYERS    = ["miRNA"] if RUN_SCALE == "quick" else \
                ["miRNA", "CNV", "Methy", "mRNA"]

print(f"SelectOmics {SelectOmics.__version__} · Python {sys.version.split()[0]}")
print(f"RUN_SCALE = {RUN_SCALE!r} · primary layer {PRIMARY_LAYER}")

## 2 · Data

The `_aligned.csv` files are features-by-samples and are transposed on load.
Column names are suffixed with their layer so the merged matrix stays
attributable, which Section 8 depends on.

In [ ]:
DATA_DIR = Path(".").resolve()
TARGET   = "Label"

# The MLOmics data is not committed to this repository. mlomics_data fetches it
# from a pinned, immutable dataset revision on first use, verifies each file
# against a recorded SHA-256, and caches it under examples/.mlomics_cache/.
# See examples/mlomics_data.py for the source, the licence and the citation.
if str(DATA_DIR.parent) not in sys.path:
    sys.path.insert(0, str(DATA_DIR.parent))
from mlomics_data import build_input

def build_layer_csv(layer: str) -> Path:
    """Fetch one layer, shape it for SelectOmics, cache to disk."""
    out = DATA_DIR / f"OV_{layer}_showcase.csv"
    if out.exists():
        return out
    build_input("OV", layer).to_csv(out, index=False)
    return out

layer_paths = {L: build_layer_csv(L) for L in ALL_LAYERS}

rows = []
for L, path in layer_paths.items():
    d = pd.read_csv(path, nrows=1)
    n_feat = d.shape[1] - 1
    n_samp = sum(1 for _ in open(path, encoding="utf-8")) - 1
    rows.append({"layer": L, "samples": n_samp, "features": n_feat,
                 "n/p": round(n_samp / n_feat, 4)})
display(pd.DataFrame(rows))

primary = pd.read_csv(layer_paths[PRIMARY_LAYER])
counts = primary[TARGET].value_counts().sort_index()
print("class balance:")
for cls, k in counts.items():
    print(f"  class {cls}: {k:>3}  ({100 * k / len(primary):4.1f}%)")
print(f"imbalance: {counts.max() / counts.min():.2f}:1")

## 3 · A run with every evaluation module on

Each flag below turns on a distinct piece of reporting. Section 4 takes them
one at a time.

Nested CV is the expensive one: each outer fold reruns the whole procedure,
Step 3 and validation included, so it adds about one full run per fold.
`outer_cv_splits=3` here rather than the default 5.

In [ ]:
config = SelectOmicsConfig(
    data_path=str(layer_paths[PRIMARY_LAYER]),
    target_column=TARGET,
    algorithm="RF",
    output_dir=f"results_showcase_{PRIMARY_LAYER}",

    n_consensus_models=5,
    quick_tune_iterations=10,
    n_bootstrap=50,
    random_seed=42,

    enable_step1=True,
    enable_step2=True,
    enable_step3=True,

    # --- the evaluation surface, every module ---
    enable_step_evaluations=True,        # per-step AUC against the reference
    enable_final_test_evaluation=True,   # held-out test metrics
    enable_nested_cv=True,               # unbiased generalisation estimate
    outer_cv_splits=3,
    calibrate_probabilities=True,        # isotonic calibration of the final model

    create_visualizations=True,
    save_intermediate_results=True,
    verbose=True,
)
print(f"output: {config.output_dir}")

In [ ]:
t0 = time.perf_counter()
pipeline = SelectOmicsPipeline(config)
results  = pipeline.run(validate=True)
elapsed  = time.perf_counter() - t0

print(f"\ncompleted in {elapsed / 60:.1f} min")
print("result keys:", sorted(results.keys()))

## 4 · The evaluation modules

### 4a · Per-step evaluation: `enable_step_evaluations`

Each step evaluates its own panel against the Step 0 reference, so you can see
whether a step helped rather than only how much it removed.

In [ ]:
rows = []
for key in ("step0", "step1", "step2", "step3"):
    r = results.get(key)
    if r is None:
        rows.append({"step": key, "status": "not run"})
        continue
    if r.get("skipped"):
        rows.append({"step": key, "status": "skipped (below Step 3's gate)"})
        continue
    ev = r.get("consensus_result") or r.get("reference_result") or {}
    rows.append({
        "step": key,
        "n_features": len(pipeline.get_selected_features(key)),
        "mean_auc": round(ev.get("mean_auc", float("nan")), 4),
        "std_auc": round(ev.get("std_auc", float("nan")), 4),
        "outcome": r.get("consensus_outcome", ""),
        "agreement": r.get("agreement_label", ""),
    })
display(pd.DataFrame(rows))

### 4b · Three validation protocols: `validate=True`

Stratified CV, leave-one-out and bootstrap, run against **every** step's panel.
They disagree in useful ways: the bootstrap interval is the one that says how
much the estimate can be trusted.

In [ ]:
comparison = results["validation"]["comparison_df"]
display(comparison.round(4))

print("\nbootstrap confidence intervals:")
for name, row in comparison.iterrows():
    lo, hi = row["bootstrap_ci_lower"], row["bootstrap_ci_upper"]
    print(f"  {name:<26} {row['bootstrap_auc']:.4f}  "
          f"[{lo:.4f}, {hi:.4f}]  width {hi - lo:.4f}  "
          f"{row.get('ci_width_flag', '')}")

### 4c · Held-out test evaluation: `enable_final_test_evaluation`

The only look at data no protocol above has seen. It scores the
**recommended** panel, the one you are told to use, and the training CV AUC it
is compared against comes from that same panel. `calibrate_probabilities`
wraps the fitted model in isotonic calibration before predicting.

In [ ]:
def _f(v):
    return "n/a" if v is None else f"{v:+.4f}" if v < 0 else f"{v:.4f}"

ft = results.get("final_test")
if ft:
    m = ft["metrics"]
    print(f"  panel scored       : {ft['panel']}, {ft['step_name']} "
          f"({ft['n_features']} features)")
    print(f"  test AUC           : {_f(ft['test_auc'])}")
    print(f"  balanced accuracy  : {_f(m.get('balanced_accuracy'))}")
    print(f"  training CV AUC    : {_f(m.get('training_cv_auc_mean'))}  "
          f"(same panel)")
    print(f"  generalisation gap : {_f(m.get('generalization_gap'))}  "
          f"({ft['gen_label']})")
    proba = np.asarray(ft["test_proba"])
    print(f"\n  probability matrix : {proba.shape}")
    print(f"  rows sum to 1      : {np.allclose(proba.sum(axis=1), 1.0)}")
    print(f"  calibrated         : {config.calibrate_probabilities}")
else:
    print("Not run. Needs run(validate=True) and "
          "enable_final_test_evaluation=True.")

### 4d · Nested cross-validation: `enable_nested_cv`

Every estimate above comes from the training split, which is also where the
features were selected and where the recommendation was made. Nested CV checks
the whole procedure on data it has not seen: each outer fold reruns selection,
validation and the recommendation on its training portion, then scores every
step's panel on the held-out fold.

Two numbers answer two questions. The headline AUC is the recommended panel's,
so it estimates what you get by following the recommendation. The regret is how
far that fell short of whichever step did best on each fold with hindsight:
zero means following the recommendation never cost anything.

It also reports how often each feature was in the recommended panel across
folds, a stability measure that does not depend on panel size.

In [ ]:
nested = results.get("nested_cv")
if nested:
    steps = [s.split(" (")[0] for s in nested["fold_recommended_steps"]]
    print(f"  outer folds          : {len(nested['fold_aucs'])}")
    print(f"  recommended per fold : {steps}")
    print(f"  held-out AUC         : {nested['mean_auc']:.4f} "
          f"+/- {nested['std_auc']:.4f}  (recommended panel)")
    print(f"  last step's panel    : {nested['last_step_mean_auc']:.4f}")
    print(f"  best step, hindsight : {nested['best_step_mean_auc']:.4f}")
    print(f"  mean regret          : {nested['mean_regret']:.4f}")

    print("\nper step, averaged over the outer folds:")
    display(nested["step_summary"].round(4))

    # Selection optimism. Validation minus held-out AUC for the recommended
    # panel, less the same gap for Step 0: Step 0 involves no selection, so
    # its gap is fold-to-fold noise, and what remains is what selection added.
    d = nested["fold_details"]
    chosen = d[d["recommended"]].set_index("fold")
    ref = d[d["step"] == "Step 0 (Reference)"].set_index("fold")
    gap_rec = (chosen["inner_score"] - chosen["outer_auc"]).mean()
    gap_ref = (ref["inner_score"] - ref["outer_auc"]).mean()
    optimism = gap_rec - gap_ref
    print(f"\n  validation minus held-out AUC: recommended {gap_rec:+.4f}, "
          f"Step 0 {gap_ref:+.4f}")
    print(f"  selection optimism (the difference): {optimism:+.4f}")
    if optimism > 0.02:
        print("  -> selection inflated the recommended panel's validation "
              "score by about this much.")
    else:
        print("  -> no meaningful selection optimism: the recommended "
              "panel validated as honestly as the unselected reference.")

    stab = nested["feature_stability"]
    print(f"\n  features in the recommended panel of every fold: "
          f"{(stab['selection_frequency'] == 1.0).sum()}")
    display(stab.head(10))
else:
    print("Not run. Set enable_nested_cv=True.")

### 4e · Sample-size adequacy

Computed before selection starts, from n, p and the smallest class. It is
advisory, and on small cohorts it is the part most worth reading.

In [ ]:
adequacy = results.get("sample_adequacy")
if adequacy:
    print(f"  overall severity : {adequacy.get('overall_severity')}")
    for k in ("n_per_p", "per_class_flag", "min_class_size"):
        if k in adequacy:
            print(f"  {k:<17}: {adequacy[k]}")
    for w in adequacy.get("warnings", []):
        print(f"    - {w}")

## 5 · The recommendation

The comparison in 4b says how each panel scored. The recommendation says which
to **use**: the smallest panel whose score is within one standard error of the
best. When a later step validates clearly worse than an earlier one, that is
the earlier panel, and you keep it.

Know its limit. Every panel is validated on the samples its features were
selected from, so a small held-out loss is invisible to it, and near-ties go
to the smaller panel. Nested CV (4d) is the check. Across seven datasets it
found that following the recommendation kept held-out AUC within about 0.01 of
using every feature (`benchmarks/BENCHMARKS.md`, section 8). The label is also
lowered when a panel scores implausibly far above the unselected Step 0, which
is what selection fitting noise looks like.

In [ ]:
rec = results["recommendation"]
print(f"  recommended step : {rec['step_id']}")
print(f"  features         : {rec['n_features']}")
print(f"  weighted AUC     : {rec['weighted_auc']:.4f}   "
      f"(0.5 x CV + 0.25 x LOO + 0.25 x bootstrap)")
print(f"  CV stability     : {rec['cv_stability']:.4f}   (std / mean)")
print(f"  quality          : {rec['quality']}")
print(f"  reason           : {rec['reason']}")

last = pipeline.get_selected_features()
recommended = pipeline.get_recommended_features()
print(f"\n  last step   : {len(last):>4} features")
print(f"  recommended : {len(recommended):>4} features")
if set(last) == set(recommended):
    print("  -> they agree: the last step is also the best-validated one.")
else:
    only_rec = set(recommended) - set(last)
    print(f"  -> they differ. The recommendation keeps {len(only_rec)} "
          f"features the last step discarded.")
    print("     Use get_recommended_features() for the validated panel.")

## 6 · Panel size against the alternatives

Every method here reaches statistically indistinguishable AUC on this kind of
data, so AUC is not the differentiator. Panel size is.

Baselines run at **library defaults**. Tuning them to a comparable ladder would
be a different experiment and would not answer what a user gets out of the box.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X_train = pipeline._X_train
y_train = pipeline._y_train
cv = StratifiedKFold(3, shuffle=True, random_state=42)

def score_panel(cols):
    """Macro OVR AUC of a fixed panel, under one shared CV scheme."""
    if len(cols) == 0:
        return float("nan")
    est = Pipeline([("s", StandardScaler()),
                    ("c", RandomForestClassifier(n_estimators=200,
                                                 random_state=42))])
    return cross_val_score(est, X_train[cols], y_train, cv=cv,
                           scoring="roc_auc_ovr").mean()

arms = {}
arms["SelectOmics (recommended)"] = list(recommended)

t = time.perf_counter()
l1 = Pipeline([("s", StandardScaler()),
               ("c", LogisticRegression(penalty="l1", solver="saga", C=1.0,
                                        max_iter=2000, random_state=42))])
l1.fit(X_train, y_train)
coef = np.abs(l1.named_steps["c"].coef_).max(axis=0)
arms["LASSO"] = list(X_train.columns[coef > 0])
lasso_s = time.perf_counter() - t

t = time.perf_counter()
en = Pipeline([("s", StandardScaler()),
               ("c", LogisticRegression(penalty="elasticnet", solver="saga",
                                        l1_ratio=0.5, C=1.0, max_iter=2000,
                                        random_state=42))])
en.fit(X_train, y_train)
coef = np.abs(en.named_steps["c"].coef_).max(axis=0)
arms["ElasticNet"] = list(X_train.columns[coef > 0])
en_s = time.perf_counter() - t

t = time.perf_counter()
sel = RFECV(RandomForestClassifier(n_estimators=100, random_state=42),
            step=max(1, X_train.shape[1] // 20), cv=cv,
            scoring="roc_auc_ovr", min_features_to_select=5)
sel.fit(X_train, y_train)
arms["RFECV"] = list(X_train.columns[sel.support_])
rfecv_s = time.perf_counter() - t

secs = {"SelectOmics (recommended)": elapsed, "LASSO": lasso_s,
        "ElasticNet": en_s, "RFECV": rfecv_s}
table = pd.DataFrame([
    {"method": k, "features": len(v), "cv_auc": round(score_panel(v), 4),
     "seconds": round(secs[k], 1)}
    for k, v in arms.items()
]).sort_values("features")
display(table)

print("Note on `seconds`: the SelectOmics figure is the complete Section 3 run,")
print("including nested CV and all three validation protocols. The baselines")
print("are selection only, so this column is not like-for-like on cost.")
print()

sm = table[table.method.str.startswith("SelectOmics")].iloc[0]
big = table[table.features == table.features.max()].iloc[0]
print(f"\n{sm.features} features vs {big.features} ({big.method}): "
      f"{big.features / max(sm.features, 1):.0f}x smaller, "
      f"AUC {sm.cv_auc:.4f} vs {big.cv_auc:.4f}")
print("\nAt full scale on TCGA LGG (10 folds, 5 layers) SelectOmics was")
print("statistically tied with the best baseline on AUC in every layer while")
print("returning 16-24 features against 64-5526: 30x to 325x smaller.")

## 7 · Step ablation

What each step contributes on its own, and whether the cascade beats its parts.
The panels are scored under the same CV scheme as Section 6.

In [ ]:
def run_arm(label, **flags):
    cfg = SelectOmicsConfig(
        data_path=str(layer_paths[PRIMARY_LAYER]), target_column=TARGET,
        algorithm="RF", output_dir=f"results_ablation_{label}",
        n_consensus_models=5, quick_tune_iterations=10, n_bootstrap=20,
        random_seed=42, verbose=False, create_visualizations=False,
        save_intermediate_results=False, enable_step_evaluations=False,
        enable_final_test_evaluation=False, **flags)
    t = time.perf_counter()
    p = SelectOmicsPipeline(cfg)
    p.run(validate=False)
    return {"arm": label, "features": len(p.get_selected_features()),
            "seconds": round(time.perf_counter() - t, 1)}

ablation = [
    run_arm("S1_only", enable_step1=True,  enable_step2=False, enable_step3=False),
    run_arm("S2_only", enable_step1=False, enable_step2=True,  enable_step3=False),
    run_arm("S1_S2",   enable_step1=True,  enable_step2=True,  enable_step3=False),
    run_arm("S1_S2_S3", enable_step1=True, enable_step2=True,  enable_step3=True),
]
abl = pd.DataFrame(ablation)
display(abl)

print("Measured across 12 synthetic scenarios at 5 seeds, where the true")
print("features are known:")
print("    S1 only    F1 0.160   precision 0.087   168 features")
print("    S2 only    F1 0.613   precision 1.000     7 features")
print("    S1 -> S2   F1 0.801   precision 0.959    10 features")
print("    full       F1 0.810   precision 0.973     9 features")
print()
n_s1s2 = abl.loc[abl.arm == "S1_S2", "features"].iloc[0]
n_full = abl.loc[abl.arm == "S1_S2_S3", "features"].iloc[0]
print(f"On THIS layer Step 2 hands over {n_s1s2} features, well above Step 3's")
print(f"gate of 30, so Step 3 runs and cuts the panel to {n_full}.")
print()
print("The synthetic figures above look like it barely helps because there")
print("Step 2 emitted around 9 features and Step 3 skipped almost every run.")
print("Both are true, and which you get depends on how much Step 2 hands over.")
print("The recommendation falls back to an earlier panel only when Step 3")
print("validates clearly worse, so a smaller cost is what nested CV measures.")
nested = results.get("nested_cv")
if nested:
    s = nested["step_summary"].set_index("step")["mean_outer_auc"]
    print(f"On this layer (4d), the recommended panel scored "
          f"{nested['mean_auc'] - s['Step 0 (Reference)']:+.3f} held-out AUC")
    print("against every feature. Across seven datasets the figure stayed")
    print("within about 0.01 (benchmarks/BENCHMARKS.md, section 8).")

## 8 · Multi-omics integration

Whether merging layers beats the best single layer. Column suffixes make the
recommended panel attributable back to its layer.

At `RUN_SCALE="quick"` only miRNA is present, so this reports the attribution
machinery rather than a real comparison. Set `RUN_SCALE="full"` for the
four-layer result.

In [ ]:
if len(ALL_LAYERS) == 1:
    print(f"Single layer ({ALL_LAYERS[0]}). Layer attribution of the "
          f"recommended panel:")
    breakdown = pd.Series(
        [c.rsplit("_", 1)[-1] for c in recommended]).value_counts()
    display(breakdown.rename("features").to_frame())
    print("\nSet RUN_SCALE='full' to compare per-layer against merged.")
else:
    per_layer = {}
    for L in ALL_LAYERS:
        cfg = SelectOmicsConfig(
            data_path=str(layer_paths[L]), target_column=TARGET,
            algorithm="RF", output_dir=f"results_layer_{L}",
            n_consensus_models=5, quick_tune_iterations=10, n_bootstrap=20,
            random_seed=42, verbose=False, create_visualizations=False,
            save_intermediate_results=False, enable_step_evaluations=True,
            enable_final_test_evaluation=False)
        p = SelectOmicsPipeline(cfg)
        p.run(validate=True)
        per_layer[L] = {"features": len(p.get_recommended_features()),
                        "step": p.results["recommendation"]["step_id"],
                        "weighted_auc": round(
                            p.results["recommendation"]["weighted_auc"], 4)}
    display(pd.DataFrame(per_layer).T)

## 9 · Reproducibility and resume

Two guarantees worth checking on your own data.

**Same seed, same panel.** Feature selection that shifts between identical runs
cannot support a claim about which features matter.

**Resume refuses stale data.** A checkpoint records a fingerprint of its source
file. Editing the data and resuming would otherwise report results computed on
the old file as though they described the new one.

In [ ]:
def panel_for(seed, tag):
    cfg = SelectOmicsConfig(
        data_path=str(layer_paths[PRIMARY_LAYER]), target_column=TARGET,
        algorithm="RF", output_dir=f"results_repro_{tag}",
        n_consensus_models=5, quick_tune_iterations=10, n_bootstrap=20,
        random_seed=seed, verbose=False, create_visualizations=False,
        save_intermediate_results=False, enable_step_evaluations=False,
        enable_final_test_evaluation=False, enable_step3=False)
    p = SelectOmicsPipeline(cfg)
    p.run(validate=False)
    return set(p.get_selected_features())

a = panel_for(42, "a")
b = panel_for(42, "b")
c = panel_for(7,  "c")

print(f"seed 42 run 1 : {len(a)} features")
print(f"seed 42 run 2 : {len(b)} features")
print(f"identical     : {a == b}")
print(f"\nseed 7        : {len(c)} features")
print(f"overlap with seed 42: {len(a & c)} of {len(a | c)} "
      f"({100 * len(a & c) / max(len(a | c), 1):.0f}%)")
print("\nA different seed giving a different panel is expected: it measures")
print("how much the selection depends on the resampling, which is what the")
print("nested-CV feature stability in 4d quantifies.")

In [ ]:
# Resume refuses a checkpoint whose source data has changed.
import shutil

probe_dir = Path("results_resume_probe")
probe_csv = Path("resume_probe.csv")
shutil.copy(layer_paths[PRIMARY_LAYER], probe_csv)

cfg = SelectOmicsConfig(
    data_path=str(probe_csv), target_column=TARGET, algorithm="RF",
    output_dir=str(probe_dir), n_consensus_models=3,
    quick_tune_iterations=5, n_bootstrap=10, verbose=False,
    create_visualizations=False, save_intermediate_results=True,
    enable_step_evaluations=False, enable_final_test_evaluation=False,
    enable_step3=False)
p = SelectOmicsPipeline(cfg)
p.load_data(); p.run_step0_reference(); p.run_step1_cleaning()
p._save_checkpoint("step1")
print(f"checkpoint written: {p._checkpoint_path().name}")

resumed = SelectOmicsPipeline(cfg)._load_checkpoint()
print(f"resume with unchanged data -> {resumed!r}")

edited = pd.read_csv(probe_csv)
edited.iloc[0, 0] = edited.iloc[0, 0] + 1.0      # one cell
edited.to_csv(probe_csv, index=False)

refused = SelectOmicsPipeline(cfg)._load_checkpoint()
print(f"resume after editing one cell -> {refused!r}")
print("\nNone means it refused and will start fresh, which is the point:")
print("resuming would have reported the old data's results as the new file's.")

## 10 · Export

In [ ]:
pipeline.save_results()
out = Path(config.output_dir)

pd.DataFrame({"feature": recommended}).to_csv(
    out / "recommended_features.csv", index=False)

provenance = pipeline.get_feature_provenance()
provenance.to_csv(out / "feature_provenance.csv", index=False)

print(f"{config.output_dir}:")
for f in sorted(out.glob("*")):
    print(f"  {f.name:<46} {f.stat().st_size / 1024:>9.1f} KB")

## Summary

| Module | Flag | Reports |
|---|---|---|
| Per-step evaluation | `enable_step_evaluations` | Each step's AUC against the reference |
| Validation | `run(validate=True)` | CV, LOO and bootstrap on every panel |
| Recommendation | `run(validate=True)` | Which panel to use, and its quality |
| Held-out test | `enable_final_test_evaluation` | Metrics on unseen data, for the recommended panel |
| Nested CV | `enable_nested_cv` | Held-out AUC of the whole procedure, whether the recommendation chose well, and cross-fold feature stability |
| Calibration | `calibrate_probabilities` | Isotonic-calibrated probabilities |
| Adequacy | always | Whether n supports the p you have |

The recommendation is the one to internalise, together with its limit. It
hands you an earlier panel when a later step validates clearly worse, and it
lowers its own label when a panel scores implausibly far above the unselected
reference. It cannot see a small held-out loss, because validation shares its
samples with selection. Nested CV (4d) can.